# DDPG + SLM Sentiment Portfolio Workflow

This notebook runs the experimental SLM extension.

The offline model is still a DDPG model. The SLM signal is added during online evaluation.


## 0. Import shared workflow API

- Keep the notebook readable.
- Keep reusable code in `src/finance_rl_slm/`.
- Use the same output folders as the pure DDPG notebook.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in (PROJECT_ROOT / "src", PROJECT_ROOT, PROJECT_ROOT / "src" / "FinRL"):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from finance_rl_slm.config import DEFAULT_CONFIG
from finance_rl_slm.workflow import (
    build_daily_sentiment,
    build_sentiment_inputs,
    load_online_price_data,
    load_price_data,
    plot_normalized_prices,
    print_runtime_context,
    result_picture_path,
    result_profile_path,
    run_slm_online,
    split_price_data,
    train_slm_model_from_price_data,
)
from finance_rl_slm.evaluation import run_online_evaluation
from finance_rl_slm.training import train_offline_model


## 1. Configure the experiment

- Online test window: `2026-01-01` to `2026-06-21`.
- Pipeline name: `ddpg_slm`.
- News sample: `news_max_items = 3` per ticker for a practical local Granite run.
- SLM sentiment is treated as an experimental extra feature.


In [ ]:
config = replace(
    DEFAULT_CONFIG,
    online_start="2026-01-01",
    online_end="2026-06-21",
    news_max_items=3,
)
print_runtime_context(config)


## 2. Run online evaluation with SLM sentiment

- This is the default safe path.
- It loads the existing `ddpg_portfolio_slm.zip` model.
- It builds weekly sentiment from a small RSS sample.
- It does not retrain the model.


In [ ]:
ddpg_slm_logs = run_slm_online(config)
ddpg_slm_logs.head()


## 3. Optional offline training

- Keep this disabled during normal notebook execution.
- Turn it on only when you intentionally want to retrain the model.
- SLM is not used during offline training.


In [ ]:
RUN_OPTIONAL_TRAINING = False

if RUN_OPTIONAL_TRAINING:
    model = train_slm_model_from_price_data(config)
else:
    print("Optional training skipped. Set RUN_OPTIONAL_TRAINING = True to retrain.")


## 4. Inspect online evaluation output

- `ddpg_slm_logs` was created by the default online cell.
- Plots and profile CSV files are saved under `addenda/`.


In [ ]:
ddpg_slm_logs.head()


## 5. Output notes

- Figure files:
  - `addenda/result_picture/online_reward_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_wealth_ddpg_slm_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_daily_return_ddpg_slm_2026-01-01_2026-06-21.png`

- Profile file:
  - `addenda/result_profile_comparse/ddpg_slm_online_profile_2026-01-01_2026-06-21.csv`

- Compare it with the pure DDPG profile using `src/tool/compare_ddpg_profiles.py`.


## Workflow Notes - DDPG + SLM

- The SLM signal is a slow sentiment feature.

- It is not a tick-level trading signal.

- The DDPG model is trained without SLM first.

- During online evaluation, the environment reads:
  - the current trading date,
  - the daily sentiment value mapped from weekly news,
  - the same price-based technical state.

- The last observation slot stores the clipped sentiment score in `[-1, 1]`.

- This pipeline is experimental.

- The key question is simple:
  - Does a coarse sentiment feature improve the long-horizon behavior?
  - Or does it add noise to the investment policy?
